# REST Countries – population, borders and official languages

This notebook downloads data about every country from the [REST Countries API](https://restcountries.com/) (v5)
and answers a set of questions about population, neighbours and official languages.

**Contents**

1. Connect to the API (with safe API-key handling)
2. Download the data for all countries
3. Save the country names
4. Countries by population (descending)
5. Country with the most neighbours
6. Top 10 official languages (by number of countries)
7. Reusable function `top_languages`
8. Top official languages weighted by population
9. Chart: how the ranking changes between 6 and 8

> **Before running:** create a `.env` file next to this notebook containing
> `RESTCOUNTRIES_API_KEY=your_key_here` (see `README.md`).

## 0. Setup

In [ ]:
# Standard library
import json
import os
import time
from pathlib import Path

# Third-party
import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

# Show full tables instead of truncated ones when we ask for many rows
pd.set_option("display.max_rows", 300)

## 1. Connect to the REST Countries API

Since 2026 the API (version 5) requires an API key, sent as a *Bearer token* in the
`Authorization` header. The older key-less endpoints (`/v3.1/all`) are retired.

### API-key best practices used here

| Practice | How it is applied in this project |
|---|---|
| Never hard-code the key in source code | The key is read from an environment variable |
| Keep secrets in a local, un-tracked file | Key lives in `.env`, which is listed in `.gitignore` |
| Document what is needed without revealing it | `.env.example` is committed with a placeholder value |
| Send the key in a header, not in the URL | URLs end up in logs, browser history and error messages |
| Never print or log the key | Only a yes/no "key loaded" message is shown |
| Fail early with a clear message | A missing key raises a readable error before any request |
| Limit the damage if it leaks | Rotate (revoke + re-create) the key in the dashboard |

In [ ]:
# --- Configuration -----------------------------------------------------------
BASE_URL = "https://api.restcountries.com/countries/v5"

# Ask the API only for the fields we need -> smaller, faster responses
RESPONSE_FIELDS = "names.common,names.official,codes.alpha_3,population,borders,languages"

PAGE_SIZE = 100          # maximum page size on the free plan
REQUEST_TIMEOUT = 30     # seconds; never call an API without a timeout

# The free plan has a monthly request quota, so the raw response is cached on disk.
# Delete this file (or set FORCE_REFRESH = True) to download fresh data.
CACHE_FILE = Path("data") / "countries_raw.json"
FORCE_REFRESH = False

In [ ]:
def get_api_key() -> str:
    """Read the API key from the environment (populated from the local .env file)."""
    # load_dotenv() copies KEY=value pairs from .env into os.environ.
    # Variables that already exist in the environment are NOT overwritten,
    # so the same code also works in CI / cloud where secrets are injected directly.
    load_dotenv()

    api_key = os.getenv("RESTCOUNTRIES_API_KEY")
    if not api_key:
        raise RuntimeError(
            "RESTCOUNTRIES_API_KEY is not set. Copy .env.example to .env "
            "and paste your key there (see README.md)."
        )
    return api_key


def create_session(api_key: str) -> requests.Session:
    """Create an HTTP session that sends the key with every request."""
    session = requests.Session()
    session.headers.update(
        {
            "Authorization": f"Bearer {api_key}",  # key travels in a header, not the URL
            "Accept": "application/json",
        }
    )
    return session

## 2. Download the data for all countries

The list endpoint is paginated (`limit` / `offset`). The response looks like:

```json
{"data": {"objects": [ ...countries... ],
          "meta": {"total": 250, "count": 100, "limit": 100, "offset": 0, "more": true}}}
```

We keep requesting pages until `meta.more` is `false`.

In [ ]:
def fetch_all_countries(session: requests.Session) -> list[dict]:
    """Download every country record, page by page."""
    countries: list[dict] = []
    offset = 0

    while True:
        params = {
            "response_fields": RESPONSE_FIELDS,
            "limit": PAGE_SIZE,
            "offset": offset,
        }
        response = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)

        # 429 = too many requests -> wait a little and retry the same page
        if response.status_code == 429:
            time.sleep(10)
            continue

        # Any other 4xx/5xx status raises an exception (401 = bad or missing key)
        response.raise_for_status()

        payload = response.json()["data"]
        countries.extend(payload["objects"])

        if not payload["meta"]["more"]:
            break
        offset += PAGE_SIZE

    return countries


def load_countries() -> list[dict]:
    """Return the raw country data, from the local cache if available."""
    if CACHE_FILE.exists() and not FORCE_REFRESH:
        print(f"Loading cached data from {CACHE_FILE}")
        return json.loads(CACHE_FILE.read_text(encoding="utf-8"))

    # The key is only needed (and only read) when we really call the API
    session = create_session(get_api_key())
    print("API key loaded - downloading data ...")   # note: the key itself is never printed
    countries = fetch_all_countries(session)

    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    CACHE_FILE.write_text(json.dumps(countries, ensure_ascii=False), encoding="utf-8")
    return countries


raw_countries = load_countries()
print(f"Number of countries downloaded: {len(raw_countries)}")

In [ ]:
# Look at one raw record to understand the structure before transforming it
raw_countries[0]

### Transform the nested JSON into a flat table

Each record is nested (`names.common`, a list of language objects, ...).
We extract only what we need into a tidy `DataFrame` with one row per country.

In [ ]:
def extract_language_names(languages) -> list[str]:
    """Return a plain list of language names from the API's `languages` field.

    v5 returns a list of objects, e.g. [{"name": "English", "bcp47": "en", ...}].
    Missing values (e.g. uninhabited territories) become an empty list.
    """
    if not languages:
        return []
    if isinstance(languages, dict):          # older API shape: {"eng": "English"}
        return list(languages.values())
    return [lang["name"] if isinstance(lang, dict) else str(lang) for lang in languages]


countries_df = pd.DataFrame(
    {
        "country": country["names"]["common"],
        "official_name": country["names"].get("official"),
        "code": (country.get("codes") or {}).get("alpha_3"),
        "population": country.get("population") or 0,
        "borders": country.get("borders") or [],
        "languages": extract_language_names(country.get("languages")),
    }
    for country in raw_countries
)

# Basic sanity checks
countries_df["population"] = countries_df["population"].astype("int64")
assert countries_df["country"].is_unique, "Country names should be unique"

countries_df.head()

## 3. Save the country names in an appropriate object

A sorted **list** is the natural choice: an ordered collection of strings that is easy to
iterate over, search and slice. (The same names are also the `country` column of the DataFrame.)

In [ ]:
country_names: list[str] = sorted(countries_df["country"])

print(f"{len(country_names)} countries")
print(country_names[:10], "...")

## 4. Countries by population size (descending)

In [ ]:
countries_by_population = (
    countries_df[["country", "population"]]
    .sort_values(["population", "country"], ascending=[False, True])  # name breaks ties
    .reset_index(drop=True)
)
countries_by_population.index += 1          # rank starts at 1 instead of 0
countries_by_population.index.name = "rank"

# Thousands separators make large numbers readable (display only, data stays numeric)
countries_by_population.style.format({"population": "{:,}"})

## 5. Country bordered by the largest number of neighbours

`borders` is a list of ISO alpha-3 codes of neighbouring countries, so the number of
neighbours is simply the length of that list. Several countries could share first place,
so we return *all* countries with the maximum value rather than only the first one.

In [ ]:
countries_df["neighbour_count"] = countries_df["borders"].str.len()

max_neighbours = countries_df["neighbour_count"].max()
most_neighbours = countries_df.loc[
    countries_df["neighbour_count"] == max_neighbours,
    ["country", "neighbour_count", "borders"],
]

# Translate neighbour codes into readable country names
code_to_name = dict(zip(countries_df["code"], countries_df["country"]))
most_neighbours = most_neighbours.assign(
    neighbours=most_neighbours["borders"].apply(
        lambda codes: ", ".join(sorted(code_to_name.get(c, c) for c in codes))
    )
).drop(columns="borders")

for row in most_neighbours.itertuples():
    print(f"{row.country} has the most neighbours: {row.neighbour_count}")
    print(f"  -> {row.neighbours}")

## 6. The 10 most common official languages

`languages` holds a list per country. `explode()` turns it into one row per
(country, language) pair, which makes counting straightforward.

In [ ]:
# One row per country-language pair; countries without any language are dropped
country_languages = (
    countries_df[["country", "population", "languages"]]
    .explode("languages")
    .dropna(subset=["languages"])
    .rename(columns={"languages": "language"})
)

top10_by_countries = (
    country_languages.groupby("language")["country"]
    .count()
    .rename("countries")
    .reset_index()
    .sort_values(["countries", "language"], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)
top10_by_countries.index += 1
top10_by_countries

## 7. Function `top_languages`

The logic from step 6 wrapped into a reusable function. `nr_of_top_languages` sets how many
languages are returned. The optional `by` argument lets the same function answer step 8 too.

In [ ]:
def top_languages(nr_of_top_languages: int = 10, by: str = "countries") -> pd.DataFrame:
    """Return the most common official languages.

    Parameters
    ----------
    nr_of_top_languages : int
        Number of top languages to return.
    by : {"countries", "population"}
        "countries"  - rank by the number of countries listing the language as official.
        "population" - rank by the total population of those countries.

    Returns
    -------
    pd.DataFrame with columns: language, countries, population (index = rank).
    """
    if nr_of_top_languages < 1:
        raise ValueError("nr_of_top_languages must be at least 1")
    if by not in ("countries", "population"):
        raise ValueError("by must be 'countries' or 'population'")

    summary = (
        country_languages.groupby("language")
        .agg(countries=("country", "count"), population=("population", "sum"))
        .reset_index()
        # language name as second key -> ties are always resolved the same way
        .sort_values([by, "language"], ascending=[False, True])
        .reset_index(drop=True)
    )
    summary.index += 1
    summary.index.name = "rank"
    return summary.head(nr_of_top_languages)


# Same result as step 6, plus a different size to show the parameter works
display(top_languages(10))
display(top_languages(nr_of_top_languages=5))

## 8. Official languages ranked by population

Assumption given in the task: *if a language is official in a country, the entire population
of that country speaks it.* A multilingual country therefore contributes its full population
to **each** of its official languages, so the totals overlap and add up to more than the
world population. The result is an upper bound, not a real speaker count.

In [ ]:
top10_by_population = top_languages(10, by="population")
top10_by_population.style.format({"population": "{:,}"})

In [ ]:
# Which languages entered / left the top 10 when we switch to population?
by_countries = set(top_languages(10, by="countries")["language"])
by_population = set(top10_by_population["language"])

print("In both lists:           ", ", ".join(sorted(by_countries & by_population)))
print("Only by country count:   ", ", ".join(sorted(by_countries - by_population)))
print("Only by population:      ", ", ".join(sorted(by_population - by_countries)))

## 9. Chart: how the ranking changes

Two views of the same comparison:

* **Bar charts** – the actual values behind each top 10.
* **Slope chart** – each language's rank in both lists, so movers are visible at a glance.

In [ ]:
TOP_N = 10
COLOR_BOTH = "#2a6fbb"   # language is in both top lists
COLOR_ONE = "#e07b39"    # language is in only one of them

top_c = top_languages(TOP_N, by="countries")
top_p = top_languages(TOP_N, by="population")
in_both = set(top_c["language"]) & set(top_p["language"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

panels = [
    (axes[0], top_c, "countries", "Number of countries", "{:,.0f}", 1),
    (axes[1], top_p, "population", "Population (billions)", "{:.2f}", 1e9),
]
for ax, data, column, xlabel, label_format, divisor in panels:
    values = data[column] / divisor
    colors = [COLOR_BOTH if lang in in_both else COLOR_ONE for lang in data["language"]]
    bars = ax.barh(data["language"], values, color=colors)
    ax.bar_label(bars, labels=[label_format.format(v) for v in values], padding=3)
    ax.invert_yaxis()                      # rank 1 on top
    ax.set_xlabel(xlabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.12)                     # room for the value labels

axes[0].set_title(f"Top {TOP_N} official languages by number of countries")
axes[1].set_title(f"Top {TOP_N} official languages by population")

# Shared legend explaining the two colours
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (COLOR_BOTH, COLOR_ONE)]
fig.legend(handles, ["In both top lists", "In this list only"],
           loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

In [ ]:
# --- Slope chart: rank by countries (left) vs rank by population (right) -------
# Full rankings, so we also know the true rank of languages outside the top N
all_languages = country_languages["language"].nunique()
rank_c = top_languages(all_languages, by="countries").reset_index().set_index("language")["rank"]
rank_p = top_languages(all_languages, by="population").reset_index().set_index("language")["rank"]

languages_to_plot = sorted(set(top_c["language"]) | set(top_p["language"]), key=rank_c.get)


def chart_positions(ranks: pd.Series) -> dict:
    """Top-N languages sit at their rank; the others are stacked below a divider."""
    positions, next_free = {}, TOP_N + 1.6
    for language in sorted(languages_to_plot, key=ranks.get):
        if ranks[language] <= TOP_N:
            positions[language] = ranks[language]
        else:
            positions[language] = next_free
            next_free += 1
    return positions


pos_c, pos_p = chart_positions(rank_c), chart_positions(rank_p)

fig, ax = plt.subplots(figsize=(9, 8))
for language in languages_to_plot:
    color = COLOR_BOTH if language in in_both else COLOR_ONE
    ax.plot([0, 1], [pos_c[language], pos_p[language]], marker="o", color=color, linewidth=2)
    # Direct labels with the true rank on both sides (no legend lookup needed)
    ax.text(-0.04, pos_c[language], f"{language}  #{rank_c[language]}", ha="right", va="center")
    ax.text(1.04, pos_p[language], f"#{rank_p[language]}  {language}", ha="left", va="center")

# Divider between "inside the top N" and "outside the top N"
ax.axhline(TOP_N + 0.8, color="grey", linestyle="--", linewidth=1)
ax.text(-0.6, TOP_N + 0.7, f"outside the top {TOP_N}", ha="left", va="bottom", color="grey", fontsize=9)

ax.set_xlim(-0.6, 1.6)
ax.invert_yaxis()                              # rank 1 on top
ax.set_xticks([0, 1], ["Rank by\nnumber of countries", "Rank by\npopulation"])
ax.set_yticks([])
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(axis="x", length=0)
ax.set_title("How the ranking of official languages changes\nwhen population is taken into account")
handles = [plt.Line2D([0], [0], color=c, marker="o", linewidth=2) for c in (COLOR_BOTH, COLOR_ONE)]
ax.legend(handles, ["In both top lists", "In only one top list"],
          loc="lower center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.show()

## Summary

* Counting **countries** favours languages spread across many (often small) states.
* Weighting by **population** favours languages of a few very populous countries.
* The population figures are an upper bound: they assume everybody in a country speaks every
  official language, and multilingual countries are counted once per language.

*Data source: [REST Countries](https://restcountries.com/).*